# 12 · Treino MLP — Classificação Binária de Alto Risco de OLA

ML-4: em vez de prever a **taxa** de violação (regressão, já feito no
`11`), prevê se um segmento vai entrar em **estado de alto risco**
(sim/não) no horizonte D+1/D+7. Mesma arquitetura de rede do `11`
(`128 → 64 → 32 → 1`), trocando a saída/perda pra classificação
(`BCEWithLogitsLoss` em vez de `MSELoss`) e a métrica de portão pra
**F1-score** em vez de MAE (MAE não faz sentido pra rótulo binário; F1
é mais informativo que acurácia pura quando as classes são
desbalanceadas, que é o esperado aqui — a maioria dos dias não deve
estar em "alto risco").

**Limiar de "alto risco"**: percentil 75 da taxa de violação **no
treino** (adaptativo ao dado, evita cravar um número arbitrário tipo
"20%") — ajustável via `PERCENTIL_ALTO_RISCO` abaixo, sem mexer no
resto da lógica.

**Baseline de comparação**: classifica "alto risco" se
`taxa_media_movel_7d` (o mesmo valor usado como baseline no `11`) já
estiver acima do limiar — mesma filosofia de "ontem parecido com
amanhã", agora em versão binária.

Não testei localmente antes de entregar (a pedido — sem gastar crédito
nessa rodada). Sintaxe validada, mas rodar de verdade fica por sua
conta primeiro.

In [0]:
%pip install -q torch
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ./00_config

# 00 · Configuração do projeto AntecipeAI

Este notebook **não é uma etapa do pipeline** — ele é chamado com `%run` no
início de todos os outros notebooks para carregar a configuração central
do projeto a partir do arquivo `.env`.

A ideia por trás disso: migrar o projeto do Databricks Free (tudo managed,
storage do próprio metastore) para um ambiente de nuvem (S3/AWS,
ADLS/Azure, GCS/GCP, Object Storage/OCI) deve ser uma **troca de valores
no `.env`**, e não uma reescrita de notebook.

## Localizar e carregar o `.env`

Assumimos a estrutura de pastas `antecipeai/notebooks/` e
`antecipeai/config/antecipeai.env` lado a lado no Repo/Workspace. Se a sua
estrutura for diferente, informe o caminho exato no widget
`env_file_path` antes de rodar este notebook.

Configuração carregada de: ../config/antecipeai.env


## Defaults (usados apenas se o `.env` não for encontrado)

## Variáveis expostas para os notebooks que derem `%run` neste

Por ser chamado via `%run`, tudo que é definido aqui fica disponível no
notebook que chamou — não precisa importar nada manualmente depois.

Configuração ativa:
  CATALOG         = antecipeai
  SCHEMA_LANDING  = landing
  SCHEMA_BRONZE   = bronze
  SCHEMA_SILVER   = silver
  SCHEMA_GOLD     = gold
  VOLUME_RAW      = raw
  TABLE_TYPE      = MANAGED
  STORAGE_ROOT    = (vazio - ok p/ MANAGED)
  CLOUD_PROVIDER  = NONE


## Helper: DDL de criação de schema (MANAGED vs EXTERNAL)

Centraliza a única parte do projeto que de fato muda entre "Databricks
Free" e "produção na nuvem": **onde** o schema grava fisicamente os
dados. O resto do código (leituras, transformações, escrita de tabelas)
não muda uma linha.

Helpers disponíveis: qualified_table(schema, table), volume_path(subpath), create_schema_sql(schema_name)


In [0]:
from pyspark.sql import functions as F
from datetime import timedelta
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

/local_disk0/.ephemeral_nfs/envs/pythonEnv-01a7bfd2-5cc2-47c1-a436-a2485d189f39/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


In [0]:
DATA_INICIO_TREINO_ML = "2024-12-01"
PERCENTIL_ALTO_RISCO = 0.75


def split_temporal_pandas(df, coluna_data="data_abertura", dias_teste=30, dias_validacao=30, dias_embargo=7):
    data_max = df[coluna_data].max()
    inicio_teste = data_max - timedelta(days=dias_teste - 1)
    fim_embargo_teste = inicio_teste - timedelta(days=dias_embargo)
    inicio_validacao = fim_embargo_teste - timedelta(days=dias_validacao - 1)
    fim_embargo_validacao = inicio_validacao - timedelta(days=dias_embargo)
    teste = df[df[coluna_data] >= inicio_teste].copy()
    validacao = df[(df[coluna_data] >= inicio_validacao) & (df[coluna_data] <= fim_embargo_teste)].copy()
    treino = df[df[coluna_data] <= fim_embargo_validacao].copy()
    return treino, validacao, teste


def construir_mlp_classificacao(n_features):
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Linear(n_features, 128), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 1),  # logit puro — sigmoid aplicado só na hora de avaliar/prever
    )


def f1_score_manual(y_true, y_pred_bin):
    tp = ((y_true == 1) & (y_pred_bin == 1)).sum()
    fp = ((y_true == 0) & (y_pred_bin == 1)).sum()
    fn = ((y_true == 1) & (y_pred_bin == 0)).sum()
    precisao = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return 2 * precisao * recall / (precisao + recall) if (precisao + recall) > 0 else 0.0


print("Funções prontas.")

Funções prontas.


In [0]:
def treinar_classificacao_com_gate(
    nome_tabela: str, target_taxa: str, coluna_categorica: str = None, epocas: int = 250, lr: float = 0.002
) -> dict:
    """
    Treina um MLP de classificação binária ("alto risco" sim/não) a partir
    de uma tabela silver.features_risco_ola_*. Mesmo padrão de portão dos
    notebooks anteriores, mas com F1-score em vez de MAE.
    """
    colunas_lag = ["taxa_lag_1d", "taxa_lag_7d", "taxa_media_movel_7d", "taxa_media_movel_14d"]
    coluna_baseline = "taxa_media_movel_7d"

    serie = spark.table(qualified_table(SCHEMA_SILVER, nome_tabela))
    calendario = spark.table(qualified_table(SCHEMA_SILVER, "features_calendario"))
    sdf = serie.join(calendario, "data_abertura", "left").filter(F.col("data_abertura") >= DATA_INICIO_TREINO_ML)
    sdf = sdf.withColumn("is_fim_de_semana_int", F.col("is_fim_de_semana").cast("int"))

    colunas_features = colunas_lag + [
        "dia_semana_num", "trimestre", "is_feriado", "is_fim_de_semana_int",
        "qtd_kpi_regra_divergente", "qtd_duracao_suspeita",
    ]
    if coluna_categorica != "prioridade_num":
        colunas_features.append("prioridade_num")

    cols_select = ["data_abertura"] + colunas_features + [target_taxa]
    if coluna_baseline not in colunas_features:
        cols_select.append(coluna_baseline)
    if coluna_categorica:
        cols_select.append(coluna_categorica)

    pdf_bruto = sdf.select(*cols_select).toPandas()
    pdf_bruto["data_abertura"] = pdf_bruto["data_abertura"].astype("datetime64[ns]")

    if coluna_categorica:
        categorias = pdf_bruto[coluna_categorica].astype("category")
        pdf_bruto[f"{coluna_categorica}_idx"] = categorias.cat.codes
        colunas_features = colunas_features + [f"{coluna_categorica}_idx"]

    pdf = pdf_bruto.dropna(subset=colunas_lag + [target_taxa])
    treino, validacao, teste = split_temporal_pandas(pdf)

    # Limiar calculado SÓ no treino — evita vazar informação de val/teste
    # para a definição do que conta como "alto risco".
    limiar = treino[target_taxa].quantile(PERCENTIL_ALTO_RISCO)

    def rotular(df):
        return (df[target_taxa] > limiar).astype(np.float32).values.reshape(-1, 1)

    y_treino_bin = rotular(treino)
    y_val_bin = rotular(validacao)
    y_teste_bin = rotular(teste)

    medias = treino[colunas_features].mean()
    desvios = treino[colunas_features].std().replace(0, 1)

    def normalizar_X(df):
        return torch.tensor(((df[colunas_features] - medias) / desvios).fillna(0).values.astype(np.float32))

    X_treino = normalizar_X(treino)
    X_val = normalizar_X(validacao)
    X_teste = normalizar_X(teste)
    y_treino_t = torch.tensor(y_treino_bin)

    # Classe "alto risco" é rara por construção (achado da EDA: >90% dos
    # dias/segmentos têm taxa de violação = 0). Sem pesar a classe rara, a
    # rede aprende o caminho preguiçoso — prever sempre "não é risco" já
    # acerta ~95% das linhas, e o F1 do modelo zera (nunca prevê positivo).
    # pos_weight contrabalança isso: erro em classe positiva pesa mais.
    n_positivos = y_treino_bin.sum()
    n_negativos = len(y_treino_bin) - n_positivos
    pos_weight = torch.tensor([n_negativos / n_positivos]) if n_positivos > 0 else torch.tensor([1.0])

    modelo = construir_mlp_classificacao(len(colunas_features))
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr, weight_decay=1e-4)
    perda_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    for _ in range(epocas):
        modelo.train()
        otimizador.zero_grad()
        perda = perda_fn(modelo(X_treino), y_treino_t)
        perda.backward()
        otimizador.step()

    modelo.eval()
    with torch.no_grad():
        pred_val_bin = (torch.sigmoid(modelo(X_val)).numpy() > 0.5).astype(np.float32)
        pred_teste_bin = (torch.sigmoid(modelo(X_teste)).numpy() > 0.5).astype(np.float32)

    f1_val_modelo = f1_score_manual(y_val_bin, pred_val_bin)
    f1_teste_modelo = f1_score_manual(y_teste_bin, pred_teste_bin)

    pred_val_baseline = (validacao[coluna_baseline].values > limiar).astype(np.float32).reshape(-1, 1)
    pred_teste_baseline = (teste[coluna_baseline].values > limiar).astype(np.float32).reshape(-1, 1)
    f1_val_baseline = f1_score_manual(y_val_bin, pred_val_baseline)
    f1_teste_baseline = f1_score_manual(y_teste_bin, pred_teste_baseline)

    vencedor = "modelo" if f1_val_modelo > f1_val_baseline else "baseline"

    ultima_linha = teste.sort_values("data_abertura").tail(1)
    X_ultima = normalizar_X(ultima_linha)
    if vencedor == "modelo":
        with torch.no_grad():
            prob_ultima = torch.sigmoid(modelo(X_ultima)).item()
    else:
        prob_ultima = float(ultima_linha[coluna_baseline].values[0] > limiar)
    dias_futuro = 1 if "d1" in target_taxa else 7
    data_prevista = (ultima_linha["data_abertura"].values[0] + np.timedelta64(dias_futuro, "D"))

    return {
        "tabela": nome_tabela, "target": target_taxa, "limiar_alto_risco": round(float(limiar), 4),
        "vencedor": vencedor,
        "f1_val_modelo": round(f1_val_modelo, 4), "f1_val_baseline": round(f1_val_baseline, 4),
        "f1_teste_modelo": round(f1_teste_modelo, 4), "f1_teste_baseline": round(f1_teste_baseline, 4),
        "taxa_positivos_treino": round(float(y_treino_bin.mean()), 4),
        "data_prevista": pd.Timestamp(data_prevista), "alto_risco_previsto": bool(prob_ultima > 0.5),
        "probabilidade": round(float(prob_ultima), 4),
        "produto": ultima_linha[coluna_categorica].values[0] if coluna_categorica == "produto" else None,
        "grupo_designado": ultima_linha[coluna_categorica].values[0] if coluna_categorica == "grupo_designado" else None,
    }


print("treinar_classificacao_com_gate() pronta.")

treinar_classificacao_com_gate() pronta.


## Treinar as 4 combinações (produto/equipe × D+1/D+7)

In [0]:
configuracoes_risco = [
    ("features_risco_ola_produto", "produto"),
    ("features_risco_ola_equipe", "grupo_designado"),
]

resultados_classificacao = {}
for nome_tabela, cat in configuracoes_risco:
    for horizonte in ["target_taxa_d1", "target_taxa_d7"]:
        chave = f"{nome_tabela}__{horizonte}"
        print(f"Treinando: {chave}")
        resultados_classificacao[chave] = treinar_classificacao_com_gate(nome_tabela, horizonte, cat)

print(f"\n{len(resultados_classificacao)} combinações treinadas.")

Treinando: features_risco_ola_produto__target_taxa_d1
Treinando: features_risco_ola_produto__target_taxa_d7
Treinando: features_risco_ola_equipe__target_taxa_d1
Treinando: features_risco_ola_equipe__target_taxa_d7

4 combinações treinadas.


## Resumo comparativo

`taxa_positivos_treino` mostra o quão desbalanceada ficou a classe
"alto risco" no treino — deveria ficar perto de `1 - PERCENTIL_ALTO_RISCO`
(≈0,25) por construção; se estiver muito diferente, vale investigar
antes de confiar no F1.

In [0]:
resumo = pd.DataFrame(list(resultados_classificacao.values()))
display(spark.createDataFrame(resumo))

qtd_modelo = sum(1 for r in resultados_classificacao.values() if r["vencedor"] == "modelo")
print(f"\nModelo (MLP) venceu em {qtd_modelo} de {len(resultados_classificacao)} combinações (métrica: F1-score).")

tabela,target,limiar_alto_risco,vencedor,f1_val_modelo,f1_val_baseline,f1_teste_modelo,f1_teste_baseline,taxa_positivos_treino,data_prevista,alto_risco_previsto,probabilidade,produto,grupo_designado
features_risco_ola_produto,target_taxa_d1,0.0,baseline,0.0,0.1587,0.12,0.1053,0.049,2025-12-31T00:00:00.000Z,false,0.0,lhvp,null
features_risco_ola_produto,target_taxa_d7,0.0,baseline,0.0741,0.1579,0.0,0.1695,0.0484,2025-12-31T00:00:00.000Z,false,0.0,lhco,null
features_risco_ola_equipe,target_taxa_d1,0.0,baseline,0.0,0.1967,0.0769,0.2381,0.0695,2025-12-31T00:00:00.000Z,false,0.0,null,Team09
features_risco_ola_equipe,target_taxa_d7,0.0,baseline,0.0952,0.1972,0.5882,0.2069,0.07,2025-12-31T00:00:00.000Z,false,0.0,null,Team14



Modelo (MLP) venceu em 0 de 4 combinações (métrica: F1-score).
